# Week 02 · Day 04 — Deployment

**Topics:** FastAPI + AsyncOpenAI · SSE streaming · Semantic cache · Serverless patterns


In [ ]:
%pip install -q fastapi uvicorn openai httpx numpy

## 1 · AsyncOpenAI vs Sync Client

The sync `OpenAI()` client blocks the event loop while waiting for the API response. `AsyncOpenAI()` releases the event loop, allowing other requests to be handled concurrently — critical for any web server.

In [ ]:
import os, asyncio, time
from openai import AsyncOpenAI, OpenAI

# Create ONE client at module level — reuse the connection pool
async_client = AsyncOpenAI(api_key=os.getenv("OPENAI_API_KEY"))
sync_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

async def async_call(prompt: str) -> str:
    r = await async_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=30,
        temperature=0,
    )
    return r.choices[0].message.content

# 5 concurrent requests vs 5 sequential
prompts = [f"Name one Python library. Reply in 5 words. ({i})" for i in range(5)]

t0 = time.perf_counter()
results = asyncio.run(asyncio.gather(*[async_call(p) for p in prompts]))
async_time = time.perf_counter() - t0

t0 = time.perf_counter()
for p in prompts:
    sync_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": p}],
        max_tokens=30,
        temperature=0,
    )
sync_time = time.perf_counter() - t0

print(f"5 concurrent async:     {async_time:.2f}s")
print(f"5 sequential sync:      {sync_time:.2f}s")
print(f"Speedup: {sync_time/async_time:.1f}x")

## 2 · FastAPI Endpoint with Async LLM Call

In [ ]:
# Write the FastAPI app to a file
app_code = '''
import os
from fastapi import FastAPI
from pydantic import BaseModel
from openai import AsyncOpenAI

app = FastAPI()
client = AsyncOpenAI(api_key=os.getenv("OPENAI_API_KEY"))  # module-level — shared

class ChatRequest(BaseModel):
    message: str
    model: str = "gpt-4o-mini"
    temperature: float = 0.7
    max_tokens: int = 500

class ChatResponse(BaseModel):
    reply: str
    model: str
    input_tokens: int
    output_tokens: int

@app.post("/chat", response_model=ChatResponse)
async def chat(req: ChatRequest):
    response = await client.chat.completions.create(
        model=req.model,
        messages=[{"role": "user", "content": req.message}],
        temperature=req.temperature,
        max_tokens=req.max_tokens,
    )
    return ChatResponse(
        reply=response.choices[0].message.content,
        model=req.model,
        input_tokens=response.usage.prompt_tokens,
        output_tokens=response.usage.completion_tokens,
    )

@app.get("/health")
async def health():
    return {"status": "ok"}
'''

with open("/tmp/llm_api.py", "w") as f:
    f.write(app_code)
print("FastAPI app written to /tmp/llm_api.py")
print("Run with: uvicorn llm_api:app --host 0.0.0.0 --port 8000")

In [ ]:
# Test the endpoint without running uvicorn — use TestClient
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel

app = FastAPI()
async_oai = AsyncOpenAI(api_key=os.getenv("OPENAI_API_KEY"))

class ChatRequest(BaseModel):
    message: str

@app.post("/chat")
async def chat(req: ChatRequest):
    r = await async_oai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": req.message}],
        max_tokens=100,
        temperature=0,
    )
    return {"reply": r.choices[0].message.content, "tokens": r.usage.total_tokens}

test_client = TestClient(app)
response = test_client.post("/chat", json={"message": "What is Python?"})
print(response.json())

## 3 · Server-Sent Events (SSE) Streaming

In [ ]:
import json as json_module
from fastapi.responses import StreamingResponse

app2 = FastAPI()

@app2.post("/stream")
async def stream_chat(req: ChatRequest):
    async def event_generator():
        async with async_oai.chat.completions.stream(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": req.message}],
            max_tokens=200,
        ) as stream:
            async for chunk in stream:
                delta = chunk.choices[0].delta.content
                if delta:
                    # SSE format: data: {...}\n\n
                    yield f"data: {json_module.dumps({'text': delta})}\n\n"
        yield "data: [DONE]\n\n"

    return StreamingResponse(
        event_generator(),
        media_type="text/event-stream",
        headers={
            "Cache-Control": "no-cache",
            "X-Accel-Buffering": "no",  # disable nginx buffering
        },
    )

# Demo the streaming endpoint
test_client2 = TestClient(app2)
print("SSE endpoint defined at POST /stream")
print("Client-side (JavaScript):")
print("""
  const es = new EventSource('/stream');
  es.onmessage = (e) => {
    if (e.data === '[DONE]') { es.close(); return; }
    const { text } = JSON.parse(e.data);
    document.getElementById('output').textContent += text;
  };
""")

## 4 · Semantic Cache

Cache LLM responses by semantic similarity, not just exact string match. A query "What is Python?" should return the cached response to "Explain Python to me".

In [ ]:
import numpy as np

class SemanticCache:
    def __init__(self, threshold: float = 0.92):
        self.threshold = threshold
        self.entries: list[dict] = []  # [{embedding, response, query}]

    def _embed(self, text: str) -> np.ndarray:
        r = sync_client.embeddings.create(input=[text], model="text-embedding-3-small")
        vec = np.array(r.data[0].embedding)
        return vec / np.linalg.norm(vec)  # normalize for cosine

    def _cosine(self, a: np.ndarray, b: np.ndarray) -> float:
        return float(np.dot(a, b))  # already normalized

    def get(self, query: str) -> str | None:
        q_emb = self._embed(query)
        for entry in self.entries:
            sim = self._cosine(q_emb, entry["embedding"])
            if sim >= self.threshold:
                print(f"  [CACHE HIT] similarity={sim:.4f}")
                return entry["response"]
        return None

    def set(self, query: str, response: str):
        self.entries.append({
            "embedding": self._embed(query),
            "query": query,
            "response": response,
        })

def cached_llm_call(query: str, cache: SemanticCache) -> tuple[str, bool]:
    cached = cache.get(query)
    if cached:
        return cached, True  # cache hit

    r = sync_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": query}],
        temperature=0,
        max_tokens=100,
    )
    response = r.choices[0].message.content
    cache.set(query, response)
    return response, False

cache = SemanticCache(threshold=0.92)

test_queries = [
    "What is Python?",
    "Explain Python to me.",        # should hit cache
    "Tell me about the Python language.",  # should hit cache
    "What is a neural network?",    # different topic
]

for q in test_queries:
    response, hit = cached_llm_call(q, cache)
    source = "CACHE" if hit else "API  "
    print(f"[{source}] {q[:45]} → {response[:50]}")

## 5 · Graceful Shutdown Pattern

In [ ]:
# Production FastAPI app with lifespan and background task tracking
production_app_pattern = '''
from contextlib import asynccontextmanager
from fastapi import FastAPI, BackgroundTasks
import asyncio

background_tasks_set = set()

@asynccontextmanager
async def lifespan(app: FastAPI):
    # Startup
    yield
    # Shutdown — wait for background tasks
    if background_tasks_set:
        print(f"Waiting for {len(background_tasks_set)} background tasks...")
        await asyncio.gather(*background_tasks_set, return_exceptions=True)

app = FastAPI(lifespan=lifespan)

@app.post("/chat")
async def chat(req: ChatRequest, background_tasks: BackgroundTasks):
    response = await async_oai.chat.completions.create(...)
    
    # Non-blocking: evaluate quality in background
    async def evaluate():
        await run_ragas_evaluation(req.message, response.choices[0].message.content)
    
    task = asyncio.ensure_future(evaluate())
    background_tasks_set.add(task)
    task.add_done_callback(background_tasks_set.discard)
    
    return {"reply": response.choices[0].message.content}
'''
print(production_app_pattern)

## Summary

- **AsyncOpenAI**: create ONE instance at module level; use across all requests.
- **Sync vs async**: 3-4x throughput improvement for concurrent requests.
- **SSE format**: `data: {json}\n\n` — two newlines end each event.
- **`X-Accel-Buffering: no`**: prevents nginx from buffering the stream.
- **Semantic cache**: cosine threshold 0.90–0.95; normalize embeddings first.
- **Lifespan**: use for startup/shutdown logic; track background tasks for graceful shutdown.

**Week 02 complete!** Next: Project notebooks.